In [174]:
import torch

In [175]:
data = torch.load('data_tensor_2.pt')

In [176]:
data.shape[0]

1749370

In [177]:
def return_accuracy(w):
    if type(w) is not torch.Tensor:
        w = torch.tensor(w, dtype=torch.float32)
    y_true = torch.zeros(data.shape[0], dtype=torch.long)
    U = torch.matmul(data, w)                
    y_pred = torch.argmax(U, dim=1)      
    correct = (U[:, 0] - U[:, 1]) > 0
    accuracy = correct.float().mean()
    return accuracy.item()

In [195]:
import numpy as np

def interpret_vector(weight_vector, normalize=True):
    """
    Interprets a weight vector for the trolley problem utility function.
    
    Args:
        weight_vector (list/array): Weights corresponding to features
        normalize (bool): Whether to normalize people weights to show relative preferences
        
    Returns:
        str: Human-readable interpretation of the moral preferences encoded in the weights
    """
    
    # Feature names for reference
    feature_names = ['Intervention', 'Barrier', 'CrossingSignal', 'Man', 'Woman', 
                    'Pregnant', 'Stroller', 'OldMan', 'OldWoman', 'Boy', 'Girl', 'Homeless', 
                    'LargeWoman', 'LargeMan', 'Criminal', 'MaleExecutive', 'FemaleExecutive', 
                    'FemaleAthlete', 'MaleAthlete', 'FemaleDoctor', 'MaleDoctor', 'Dog', 'Cat']
    
    if len(weight_vector) != len(feature_names):
        raise ValueError(f"Weight vector length ({len(weight_vector)}) doesn't match feature count ({len(feature_names)})")
    
    weights = np.array(weight_vector)
    
    # Separate structural and people weights
    structural_weights = weights[:3]
    people_weights = weights[3:]
    people_names = feature_names[3:]
    
    # Normalize people weights if requested
    if normalize and len(people_weights) > 0:
        people_sum = np.sum(np.abs(people_weights))
        if people_sum > 0:
            normalized_people = people_weights / people_sum
        else:
            normalized_people = people_weights
    else:
        normalized_people = people_weights
    
    output_lines = []
    output_lines.append("=== TROLLEY PROBLEM MORAL PREFERENCES ===\n")
    
    # Analyze structural preferences
    interv_weight = structural_weights[0]
    barrier_weight = structural_weights[1]
    crossing_weight = structural_weights[2]
    
    structural_insights = []
    
    if abs(interv_weight) > 0.05:
        if interv_weight > 0:
            structural_insights.append(f"Favors taking action over inaction (weight: {interv_weight:.2f})")
        else:
            structural_insights.append(f"Prefers inaction over intervention (weight: {interv_weight:.2f})")
    
    if abs(barrier_weight) > 0.05:
        if barrier_weight > 0:
            structural_insights.append(f"Values passengers over pedestrians (weight: {barrier_weight:.2f})")
        else:
            structural_insights.append(f"Values pedestrians over passengers (weight: {barrier_weight:.2f})")
    
    if abs(crossing_weight) > 0.05:
        if crossing_weight > 0:
            structural_insights.append(f"Strongly considers legal vs illegal crossing (weight: {crossing_weight:.2f})")
        else:
            structural_insights.append(f"Penalizes legal behavior - counterintuitive (weight: {crossing_weight:.2f})")
    
    if structural_insights:
        output_lines.append("STRUCTURAL PREFERENCES:")
        for insight in structural_insights:
            output_lines.append(f"  • {insight}")
        output_lines.append("")
    
    # Analyze people preferences
    if len(normalized_people) > 0:
        # Sort by preference strength
        sorted_indices = np.argsort(normalized_people)[::-1]  # Descending order
        
        # Group into categories
        highly_valued = []
        moderately_valued = []
        devalued = []
        neutral = []
        
        for idx in sorted_indices:
            name = people_names[idx]
            weight = normalized_people[idx]
            
            if weight > 0.08:  # Highly valued
                highly_valued.append((name, weight))
            elif weight > 0.05:  # Moderately valued
                moderately_valued.append((name, weight))
            else:  # Devalued
                devalued.append((name, weight))
        
        output_lines.append("PERSON TYPE PREFERENCES:")
        
        if highly_valued:
            output_lines.append("  HIGHLY VALUED:")
            for name, weight in highly_valued:
                output_lines.append(f"    • {name}: {weight:.3f}")
        
        if moderately_valued:
            output_lines.append("  MODERATELY VALUED:")
            for name, weight in moderately_valued:
                output_lines.append(f"    • {name}: {weight:.3f}")
        
        if devalued:
            output_lines.append("  DEVALUED:")
            for name, weight in devalued:
                output_lines.append(f"    • {name}: {weight:.3f}")
        
        output_lines.append("")
    
    # Generate key insights
    insights = []
    
    # Check for demographic patterns
    children_weights = [normalized_people[people_names.index(name)] for name in ['Boy', 'Girl', 'Stroller'] if name in people_names]
    adult_weights = [normalized_people[people_names.index(name)] for name in ['Man', 'Woman'] if name in people_names]
    elderly_weights = [normalized_people[people_names.index(name)] for name in ['OldMan', 'OldWoman'] if name in people_names]
    
    if children_weights and adult_weights:
        avg_children = np.mean(children_weights)
        avg_adults = np.mean(adult_weights)
        if avg_children > avg_adults + 0.02:
            insights.append(f"Shows strong preference for children over adults (children: {avg_children:.3f}, adults: {avg_adults:.3f})")
        elif avg_adults > avg_children + 0.02:
            insights.append(f"Prioritizes adults over children (adults: {avg_adults:.3f}, children: {avg_children:.3f})")
    
    # Check professional bias
    professionals = ['FemaleDoctor', 'MaleDoctor', 'FemaleExecutive', 'MaleExecutive']
    prof_weights = [normalized_people[people_names.index(name)] for name in professionals if name in people_names]
    if prof_weights:
        avg_prof = np.mean(prof_weights)
        if avg_prof > 0.06:
            insights.append(f"Values professionals highly (avg: {avg_prof:.3f})")
    
    # Check for gender bias
    male_types = ['Man', 'OldMan', 'Boy', 'LargeMan', 'MaleExecutive', 'MaleAthlete', 'MaleDoctor']
    female_types = ['Woman', 'OldWoman', 'Girl', 'LargeWoman', 'FemaleExecutive', 'FemaleAthlete', 'FemaleDoctor']
    
    male_weights = [normalized_people[people_names.index(name)] for name in male_types if name in people_names]
    female_weights = [normalized_people[people_names.index(name)] for name in female_types if name in people_names]
    
    if male_weights and female_weights:
        avg_male = np.mean(male_weights)
        avg_female = np.mean(female_weights)
        if abs(avg_male - avg_female) > 0.02:
            if avg_male > avg_female:
                insights.append(f"Shows slight male bias (male: {avg_male:.3f}, female: {avg_female:.3f})")
            else:
                insights.append(f"Shows slight female bias (female: {avg_female:.3f}, male: {avg_male:.3f})")
    
    # Check for vulnerable populations
    vulnerable = ['Homeless', 'Criminal']
    vuln_weights = [normalized_people[people_names.index(name)] for name in vulnerable if name in people_names]
    if vuln_weights:
        avg_vuln = np.mean(vuln_weights)
        if avg_vuln < -0.01:
            insights.append(f"Devalues marginalized groups (avg: {avg_vuln:.3f})")
    
    # Check animal consideration
    animals = ['Dog', 'Cat']
    animal_weights = [normalized_people[people_names.index(name)] for name in animals if name in people_names]
    if animal_weights:
        avg_animal = np.mean(animal_weights)
        if avg_animal > 0.02:
            insights.append(f"Assigns significant value to animal lives (avg: {avg_animal:.3f})")
    
    if insights:
        output_lines.append("KEY MORAL INSIGHTS:")
        for insight in insights:
            output_lines.append(f"  • {insight}")
        output_lines.append("")
    
    # Overall moral philosophy summary
    output_lines.append("MORAL PHILOSOPHY SUMMARY:")
    
    # Determine overall approach
    if interv_weight > 0.1:
        output_lines.append("  • Utilitarian approach - willing to actively intervene to maximize outcomes")
    elif interv_weight < -0.1:
        output_lines.append("  • Deontological approach - prefers not to actively cause harm")
    else:
        output_lines.append("  • Mixed approach to intervention vs. inaction")
    
    if crossing_weight > 0.1:
        output_lines.append("  • Rule-based ethics - values legal and social compliance")
    
    # Find the most extreme preference
    if len(normalized_people) > 0:
        max_idx = np.argmax(np.abs(normalized_people))
        max_weight = normalized_people[max_idx]
        max_name = people_names[max_idx]
        
        if max_weight > 0:
            output_lines.append(f"  • Strongest positive bias: {max_name} (weight: {max_weight:.3f})")
        else:
            output_lines.append(f"  • Strongest negative bias: {max_name} (weight: {max_weight:.3f})")
    
    output_lines.append(f"\nUtility Calculation: Higher scores = more likely to save that option")
    output_lines.append("Decision Rule: Choose the option (save vs. alternative) with higher utility score")
    
    return "\n".join(output_lines)


In [179]:
# Openai

vector = [-0.3, -0.5, 0.4, 1.0, 1.0, 1.6, 1.8, 0.7, 0.7, 1.5, 1.5, 0.5, 0.8, 0.8, -0.6, 1.2, 1.2, 1.4, 1.4, 1.3, 1.3, 0.6, 0.6]
print(return_accuracy(vector))
print(interpret_vector(vector))

0.7666988968849182
=== TROLLEY PROBLEM MORAL PREFERENCES ===

STRUCTURAL PREFERENCES:
  • Prefers inaction over intervention (weight: -0.30)
  • Values pedestrians over passengers (weight: -0.50)
  • Strongly considers legal vs illegal crossing (weight: 0.40)

PERSON TYPE PREFERENCES:
  HIGHLY VALUED:
    • Stroller: 0.084
  MODERATELY VALUED:
    • Pregnant: 0.074
    • Girl: 0.070
    • Boy: 0.070
    • MaleAthlete: 0.065
    • FemaleAthlete: 0.065
    • MaleDoctor: 0.060
    • FemaleDoctor: 0.060
    • FemaleExecutive: 0.056
    • MaleExecutive: 0.056
    • Man: 0.047
    • Woman: 0.047
    • LargeWoman: 0.037
    • LargeMan: 0.037
    • OldWoman: 0.033
    • OldMan: 0.033

KEY MORAL INSIGHTS:
  • Shows strong preference for children over adults (children: 0.074, adults: 0.047)
  • Assigns significant value to animal lives (avg: 0.028)

MORAL PHILOSOPHY SUMMARY:
  • Deontological approach - prefers not to actively cause harm
  • Rule-based ethics - values legal and social compliance

# Method 1 : Random Sampling

In [180]:
import numpy as np
import time

def random_search_optimizer(n_dimensions=23, n_iterations=10000, bounds=(-10.0, 10.0)):
    """
    Performs a random search to find the best weight vector.

    Args:
        n_dimensions (int): The number of dimensions in the weight vector.
        n_iterations (int): The number of random vectors to test.
        bounds (tuple): A tuple (min_val, max_val) for the random weights.

    Returns:
        tuple: A tuple containing (best_weights, best_accuracy).
    """
    print("--- Starting Random Search ---")
    start_time = time.time()
    
    best_accuracy = -1.0
    best_weights = None

    for i in range(n_iterations):
        # Generate a random weight vector within the specified bounds
        weights = np.random.uniform(bounds[0], bounds[1], n_dimensions)
        
        # Get the accuracy for this vector
        accuracy = return_accuracy(weights)
        
        # If it's the best we've seen, save it
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_weights = weights
            print(f"Iteration {i+1}/{n_iterations}: New best accuracy = {accuracy:.6f}")

    end_time = time.time()
    print(f"--- Random Search Finished in {end_time - start_time:.2f} seconds ---")
    return best_weights, best_accuracy

In [181]:
random_weights, accuracy = random_search_optimizer()

--- Starting Random Search ---
Iteration 1/10000: New best accuracy = 0.432305
Iteration 2/10000: New best accuracy = 0.451684
Iteration 3/10000: New best accuracy = 0.554182
Iteration 15/10000: New best accuracy = 0.586685
Iteration 26/10000: New best accuracy = 0.607905
Iteration 44/10000: New best accuracy = 0.634600
Iteration 95/10000: New best accuracy = 0.664774
Iteration 191/10000: New best accuracy = 0.667120
Iteration 384/10000: New best accuracy = 0.692304
Iteration 2069/10000: New best accuracy = 0.717438
--- Random Search Finished in 151.46 seconds ---


In [182]:
s = interpret_vector(random_weights)
print(s)

=== TROLLEY PROBLEM MORAL PREFERENCES ===

STRUCTURAL PREFERENCES:
  • Prefers inaction over intervention (weight: -3.39)
  • Values pedestrians over passengers (weight: -2.06)
  • Strongly considers legal vs illegal crossing (weight: 1.44)

PERSON TYPE PREFERENCES:
  HIGHLY VALUED:
    • FemaleExecutive: 0.099
    • OldWoman: 0.095
    • Woman: 0.082
  MODERATELY VALUED:
    • MaleDoctor: 0.078
    • LargeMan: 0.076
    • Pregnant: 0.073
    • Stroller: 0.064
    • Man: 0.060
    • FemaleAthlete: 0.054
    • MaleAthlete: 0.053
    • Boy: 0.031
    • Dog: 0.030
  DEVALUED:
    • Cat: -0.084

KEY MORAL INSIGHTS:
  • Prioritizes adults over children (adults: 0.071, children: 0.023)

MORAL PHILOSOPHY SUMMARY:
  • Deontological approach - prefers not to actively cause harm
  • Rule-based ethics - values legal and social compliance
  • Strongest positive bias: FemaleExecutive (weight: 0.099)

Utility Calculation: Higher scores = more likely to save that option
Decision Rule: Choose the opti

# Method 2 - Hill Climbing

In [183]:
import numpy as np
import time

# (Mock return_accuracy function would be here)

def hill_climbing_optimizer(n_dimensions=23, n_iterations=15000, bounds=(-10.0, 10.0), step_size=0.01, initial_weights=None):
    """
    Performs a hill climbing search.

    Args:
        n_dimensions (int): The number of dimensions.
        n_iterations (int): The number of iterations to try and find a better neighbor.
        bounds (tuple): A tuple (min_val, max_val) for the weights.
        step_size (float): The magnitude of the random change at each step.

    Returns:
        tuple: A tuple containing (best_weights, best_accuracy).
    """
    print("--- Starting Hill Climbing ---")
    start_time = time.time()
    
    # 1. Start with a random vector
    if initial_weights is not None:
        current_weights = np.array(initial_weights)
    else:
        current_weights = np.random.uniform(bounds[0], bounds[1], n_dimensions)
    current_accuracy = return_accuracy(current_weights)
    print(f"Initial accuracy: {current_accuracy:.6f}")

    # 2. Loop and try to find better neighbors
    for i in range(n_iterations):
        # Create a new vector by making a small, random change (a "step")
        noise = np.random.normal(0, step_size, n_dimensions)
        new_weights = current_weights + noise
        
        # Clip the values to stay within the defined bounds
        new_weights = np.clip(new_weights, bounds[0], bounds[1])

        # Evaluate the new vector
        new_accuracy = return_accuracy(new_weights)
        
        # If the new vector is better, move to that position
        if new_accuracy > current_accuracy:
            current_weights = new_weights
            current_accuracy = new_accuracy
            print(f"Iteration {i+1}/{n_iterations}: New best accuracy = {current_accuracy:.6f}")

    end_time = time.time()
    print(f"--- Hill Climbing Finished in {end_time - start_time:.2f} seconds ---")
    return current_weights, current_accuracy

In [184]:
hill_weights, accuracy = hill_climbing_optimizer(initial_weights=random_weights)
print(f"Best accuracy found: {accuracy:.6f}")

--- Starting Hill Climbing ---
Initial accuracy: 0.717438
Iteration 1/15000: New best accuracy = 0.717453
Iteration 2/15000: New best accuracy = 0.717600
Iteration 4/15000: New best accuracy = 0.717637
Iteration 6/15000: New best accuracy = 0.717674
Iteration 8/15000: New best accuracy = 0.717817
Iteration 10/15000: New best accuracy = 0.718078
Iteration 11/15000: New best accuracy = 0.718153
Iteration 13/15000: New best accuracy = 0.718195
Iteration 16/15000: New best accuracy = 0.718224
Iteration 17/15000: New best accuracy = 0.718231
Iteration 20/15000: New best accuracy = 0.718266
Iteration 21/15000: New best accuracy = 0.718268
Iteration 23/15000: New best accuracy = 0.718287
Iteration 24/15000: New best accuracy = 0.718305
Iteration 25/15000: New best accuracy = 0.718367
Iteration 26/15000: New best accuracy = 0.718452
Iteration 27/15000: New best accuracy = 0.718506
Iteration 28/15000: New best accuracy = 0.718541
Iteration 31/15000: New best accuracy = 0.718609
Iteration 32/150

In [185]:
s = interpret_vector(hill_weights)
print(s)

=== TROLLEY PROBLEM MORAL PREFERENCES ===

STRUCTURAL PREFERENCES:
  • Favors taking action over inaction (weight: 0.69)
  • Values pedestrians over passengers (weight: -1.03)
  • Strongly considers legal vs illegal crossing (weight: 3.88)

PERSON TYPE PREFERENCES:
  HIGHLY VALUED:
    • Stroller: 0.082
    • Pregnant: 0.081
  MODERATELY VALUED:
    • MaleDoctor: 0.070
    • Boy: 0.068
    • FemaleExecutive: 0.068
    • Girl: 0.066
    • FemaleDoctor: 0.059
    • Woman: 0.058
    • FemaleAthlete: 0.055
    • MaleAthlete: 0.055
    • Man: 0.054
    • OldWoman: 0.050
    • LargeMan: 0.048
    • LargeWoman: 0.047
    • MaleExecutive: 0.044
    • Homeless: 0.035
    • OldMan: 0.031

KEY MORAL INSIGHTS:
  • Values professionals highly (avg: 0.060)

MORAL PHILOSOPHY SUMMARY:
  • Utilitarian approach - willing to actively intervene to maximize outcomes
  • Rule-based ethics - values legal and social compliance
  • Strongest positive bias: Stroller (weight: 0.082)

Utility Calculation: Higher 

# Method 3: Genetic Algorithm

In [186]:
import random
import time
import numpy as np
from deap import base, creator, tools, algorithms

# (Mock return_accuracy function would be here)

# --- DEAP Setup ---
# We are trying to MAXIMIZE accuracy, so weights are 1.0
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

# Attribute generator: each weight is a float between -10 and 10
N_DIMENSIONS = 23
BOUND_LOW, BOUND_UP = -10.0, 10.0
toolbox.register("attr_float", random.uniform, BOUND_LOW, BOUND_UP)

# Structure initializers
# An "Individual" is a list of N_DIMENSIONS floats
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_float, n=N_DIMENSIONS)
# A "population" is a list of individuals
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# --- Genetic Operators ---
# Evaluation function
def eval_accuracy(individual):
    # DEAP works with lists, but our function expects a numpy array
    weights = np.array(individual)
    return (return_accuracy(weights),) # Must return a tuple

toolbox.register("evaluate", eval_accuracy)
# Crossover operator
toolbox.register("mate", tools.cxTwoPoint)
# Mutation operator
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=0.1, indpb=0.1)
# Selection operator
toolbox.register("select", tools.selTournament, tournsize=3)

def genetic_algorithm_optimizer(pop_size=200, n_generations=200):
    """
    Uses a Genetic Algorithm to find the best weight vector.
    """
    print("--- Starting Genetic Algorithm ---")
    start_time = time.time()

    pop = toolbox.population(n=pop_size)
    # Track the best individual found
    hof = tools.HallOfFame(1)
    # Gather statistics
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)

    # Run the evolutionary algorithm
    algorithms.eaSimple(pop, toolbox, cxpb=0.5, mutpb=0.2, ngen=n_generations, 
                        stats=stats, halloffame=hof, verbose=True)

    end_time = time.time()
    print(f"--- Genetic Algorithm Finished in {end_time - start_time:.2f} seconds ---")
    
    best_individual = hof[0]
    best_accuracy = best_individual.fitness.values[0]
    best_weights = np.array(best_individual)
    
    return best_weights, best_accuracy

In [187]:
genetic_weights, accuracy = genetic_algorithm_optimizer()

--- Starting Genetic Algorithm ---
gen	nevals	avg     	std      	min    	max    
0  	200   	0.496976	0.0650834	0.29169	0.68229
1  	123   	0.553647	0.0489226	0.435428	0.702875
2  	133   	0.598357	0.0450302	0.496141	0.721695
3  	120   	0.638987	0.0407054	0.506936	0.733936
4  	117   	0.672157	0.0291551	0.582866	0.733936
5  	121   	0.694279	0.0189588	0.640223	0.750771
6  	122   	0.708673	0.0162223	0.65271 	0.750771
7  	115   	0.722449	0.014101 	0.676629	0.756382
8  	120   	0.733904	0.0125016	0.699113	0.763998
9  	121   	0.74463 	0.0109021	0.705126	0.764718
10 	120   	0.753936	0.00643336	0.715438	0.769003
11 	127   	0.758407	0.00475582	0.739947	0.770505
12 	125   	0.762341	0.00426016	0.748726	0.772455
13 	112   	0.76593 	0.00380199	0.753388	0.77317 
14 	133   	0.769141	0.0029066 	0.756672	0.773151
15 	126   	0.771472	0.0013136 	0.767818	0.773648
16 	107   	0.772466	0.000671825	0.770154	0.774001
17 	122   	0.772984	0.000531171	0.771234	0.774598
18 	132   	0.773403	0.000438706	0.772368	0.7747

In [188]:
print(interpret_vector(genetic_weights))

=== TROLLEY PROBLEM MORAL PREFERENCES ===

STRUCTURAL PREFERENCES:
  • Favors taking action over inaction (weight: 1.23)
  • Values pedestrians over passengers (weight: -1.00)
  • Strongly considers legal vs illegal crossing (weight: 4.60)

PERSON TYPE PREFERENCES:
  HIGHLY VALUED:
    • Girl: 0.082
    • Pregnant: 0.082
  MODERATELY VALUED:
    • MaleDoctor: 0.070
    • Stroller: 0.070
    • FemaleDoctor: 0.067
    • Boy: 0.066
    • Woman: 0.064
    • LargeWoman: 0.056
    • Man: 0.056
    • FemaleAthlete: 0.050
    • MaleExecutive: 0.047
    • OldMan: 0.046
    • LargeMan: 0.045
    • MaleAthlete: 0.045
    • FemaleExecutive: 0.044
    • Homeless: 0.039
    • OldWoman: 0.037

MORAL PHILOSOPHY SUMMARY:
  • Utilitarian approach - willing to actively intervene to maximize outcomes
  • Rule-based ethics - values legal and social compliance
  • Strongest positive bias: Girl (weight: 0.082)

Utility Calculation: Higher scores = more likely to save that option
Decision Rule: Choose the opt

# Method 4 - Bayesian Optimization

In [189]:
import numpy as np
import time
from skopt import gp_minimize
from skopt.space import Real


def bayesian_optimizer(n_dimensions=23, n_calls=200, bounds=(-10.0, 10.0)):
    """
    Uses Bayesian Optimization to find the best weight vector.

    Args:
        n_dimensions (int): The number of dimensions.
        n_calls (int): The number of times to call return_accuracy.
        bounds (tuple): A tuple (min_val, max_val) for the weights.

    Returns:
        tuple: A tuple containing (best_weights, best_accuracy).
    """
    print("--- Starting Bayesian Optimization ---")
    start_time = time.time()
    
    # 1. Define the search space
    search_space = [Real(bounds[0], bounds[1], name=f'w_{i}') for i in range(n_dimensions)]

    # 2. Define the objective function
    # scikit-optimize performs MINIMIZATION, so we must return 1.0 - accuracy.
    def objective_function(weights):
        weights = np.array(weights)
        accuracy = return_accuracy(weights)
        return 1.0 - accuracy

    # 3. Run the optimizer
    result = gp_minimize(
        func=objective_function,
        dimensions=search_space,
        n_calls=n_calls,
        n_initial_points=50, # How many random points to probe before building the model
        random_state=42,
        verbose=True
    )

    end_time = time.time()
    print(f"--- Bayesian Optimization Finished in {end_time - start_time:.2f} seconds ---")
    
    # The result.x contains the best parameters found
    best_weights = np.array(result.x)
    # The result.fun contains the best *minimized* value (1.0 - accuracy)
    best_accuracy = 1.0 - result.fun
    
    return best_weights, best_accuracy


In [190]:
bayes_weights, accuracy = bayesian_optimizer()

--- Starting Bayesian Optimization ---
Iteration No: 1 started. Evaluating function at random point.
Iteration No: 1 ended. Evaluation done at random point.
Time taken: 0.0281
Function value obtained: 0.5235
Current minimum: 0.5235
Iteration No: 2 started. Evaluating function at random point.
Iteration No: 2 ended. Evaluation done at random point.
Time taken: 0.0166
Function value obtained: 0.4810
Current minimum: 0.4810
Iteration No: 3 started. Evaluating function at random point.
Iteration No: 3 ended. Evaluation done at random point.
Time taken: 0.0187
Function value obtained: 0.3984
Current minimum: 0.3984
Iteration No: 4 started. Evaluating function at random point.
Iteration No: 4 ended. Evaluation done at random point.
Time taken: 0.0177
Function value obtained: 0.5348
Current minimum: 0.3984
Iteration No: 5 started. Evaluating function at random point.
Iteration No: 5 ended. Evaluation done at random point.
Time taken: 0.0178
Function value obtained: 0.4687
Current minimum: 0.3

In [191]:
print(interpret_vector(bayes_weights))

=== TROLLEY PROBLEM MORAL PREFERENCES ===

STRUCTURAL PREFERENCES:
  • Favors taking action over inaction (weight: 1.45)
  • Values pedestrians over passengers (weight: -2.10)
  • Strongly considers legal vs illegal crossing (weight: 7.24)

PERSON TYPE PREFERENCES:
  MODERATELY VALUED:
    • Man: 0.058
    • Boy: 0.058
    • MaleDoctor: 0.058
    • FemaleDoctor: 0.058
    • MaleAthlete: 0.058
    • FemaleAthlete: 0.058
    • FemaleExecutive: 0.058
    • MaleExecutive: 0.058
    • Criminal: 0.058
    • Woman: 0.058
    • Pregnant: 0.058
    • Stroller: 0.058
    • Girl: 0.058
    • LargeWoman: 0.050
    • LargeMan: 0.049
    • Homeless: 0.045
    • OldWoman: 0.044
    • OldMan: 0.042

MORAL PHILOSOPHY SUMMARY:
  • Utilitarian approach - willing to actively intervene to maximize outcomes
  • Rule-based ethics - values legal and social compliance
  • Strongest positive bias: Man (weight: 0.058)

Utility Calculation: Higher scores = more likely to save that option
Decision Rule: Choose the

# Method 5 - CMA-ES

In [192]:
import numpy as np
import cma
import time


def cma_es_optimizer(n_dimensions=23, initial_guess=None, sigma=0.5, n_iterations=1000):
    """
    Uses the CMA-ES algorithm to find the best weight vector.

    Args:
        n_dimensions (int): The number of dimensions.
        initial_guess (np.ndarray): An initial starting point for the search. If None, starts at all zeros.
        sigma (float): The initial standard deviation (step size). This is a crucial parameter.
        n_iterations (int): The maximum number of iterations (function evaluations).

    Returns:
        tuple: A tuple containing (best_weights, best_accuracy).
    """
    print("--- Starting CMA-ES ---")
    start_time = time.time()

    if initial_guess is None:
        initial_guess = np.zeros(n_dimensions)

    # CMA-ES aims to MINIMIZE a function. Because our function returns
    # accuracy (higher is better), we need to minimize its negative.
    # We create a wrapper "loss" function for this purpose.
    def loss_function(weights):
        accuracy = return_accuracy(weights)
        return -accuracy # Return the negative, so minimizing it maximizes accuracy

    # Create a CMA-ES optimizer instance
    # The 'inopts' dictionary allows you to set options like bounds.
    es = cma.CMAEvolutionStrategy(initial_guess, sigma, {'bounds': [-10, 10]})

    # The optimization loop
    # We can use the 'ask-and-tell' interface for more control.
    iterations = 0
    while not es.stop() and iterations < n_iterations:
        # Ask for a new population of solutions
        solutions = es.ask()
        
        # Tell the optimizer the fitness (loss) of each solution
        # This requires calling your function for each candidate vector
        fitness_values = [loss_function(w) for w in solutions]
        es.tell(solutions, fitness_values)
        
        # Log progress and increment counter
        es.disp()
        iterations += 1

    end_time = time.time()
    print(f"--- CMA-ES Finished in {end_time - start_time:.2f} seconds ---")

    # The result is stored in the `result` property of the optimizer
    best_weights = es.result.xbest
    # The best fitness is the minimized loss (a negative number)
    best_loss = es.result.fbest
    # Convert it back to accuracy
    best_accuracy = -best_loss

    return best_weights, best_accuracy


In [193]:
cma_weights, accuracy = cma_es_optimizer()

--- Starting CMA-ES ---
(6_w,13)-aCMA-ES (mu_w=4.0,w_1=38%) in dimension 23 (seed=583088, Wed Sep 10 19:29:27 2025)
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1     13 -5.832957029342651e-01 1.0e+00 4.60e-01  5e-01  5e-01 0:00.4
    2     26 -6.880425810813904e-01 1.1e+00 4.54e-01  4e-01  5e-01 0:00.6
    3     39 -6.984823346138000e-01 1.1e+00 4.61e-01  5e-01  5e-01 0:00.9
   18    234 -7.735155820846558e-01 1.6e+00 5.44e-01  5e-01  6e-01 0:04.0
   38    494 -7.854542136192322e-01 2.4e+00 4.86e-01  4e-01  5e-01 0:08.1
   64    832 -7.869250178337097e-01 2.8e+00 1.80e-01  1e-01  2e-01 0:13.2
   96   1248 -7.874737977981567e-01 3.1e+00 7.96e-02  5e-02  9e-02 0:19.3
  100   1300 -7.874846458435059e-01 3.2e+00 6.63e-02  4e-02  7e-02 0:20.1
  142   1846 -7.876418232917786e-01 4.1e+00 2.14e-02  1e-02  2e-02 0:28.1
  191   2483 -7.876898646354675e-01 5.4e+00 6.93e-03  4e-03  7e-03 0:37.3
  200   2600 -7.876915931701660e-01 5.6e+00 5.12e-03  3e-03  6e-03 0:38.

In [196]:
print(interpret_vector(cma_weights))

=== TROLLEY PROBLEM MORAL PREFERENCES ===

STRUCTURAL PREFERENCES:
  • Favors taking action over inaction (weight: 0.43)
  • Values pedestrians over passengers (weight: -0.77)
  • Strongly considers legal vs illegal crossing (weight: 2.39)

PERSON TYPE PREFERENCES:
  HIGHLY VALUED:
    • Stroller: 0.089
    • Pregnant: 0.087
  MODERATELY VALUED:
    • Girl: 0.079
    • Boy: 0.079
    • FemaleDoctor: 0.065
    • MaleDoctor: 0.064
    • FemaleAthlete: 0.055
    • FemaleExecutive: 0.052
    • LargeWoman: 0.051
    • Woman: 0.050
  DEVALUED:
    • MaleAthlete: 0.049
    • MaleExecutive: 0.049
    • Man: 0.045
    • LargeMan: 0.043
    • Homeless: 0.039
    • OldMan: 0.038
    • OldWoman: 0.037
    • Dog: 0.013
    • Cat: 0.010
    • Criminal: 0.005

KEY MORAL INSIGHTS:
  • Shows strong preference for children over adults (children: 0.082, adults: 0.048)

MORAL PHILOSOPHY SUMMARY:
  • Utilitarian approach - willing to actively intervene to maximize outcomes
  • Rule-based ethics - values le